# 02 · Exploratory data analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NamanArora332/ENGR_418_Team_3_Wildfire_Prediction/blob/main/notebooks/02_eda.ipynb)

**Owner:** Role A · **Needs:** `fires_clean.parquet` from notebook 01 · **Runtime:** about 1 minute

Charts and tables for the report's **Data** section, plus a first look at which features separate large fires.
Figures are saved to `ENGR418_wildfire_data/figures/` in the shared Drive.

In [ ]:
# --- Setup: works on Google Colab and on your own computer ---------------------
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/NamanArora332/ENGR_418_Team_3_Wildfire_Prediction.git"
BRANCH = "main"   # change this if you want to test code from another branch
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/ENGR_418_Team_3_Wildfire_Prediction")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src" / "wildfire").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo / "src"))
print("Using code from:", repo)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wildfire import CLASS_NAMES, feature_columns, load_clean
from wildfire.config import default_data_root

DATA_ROOT = default_data_root()
FIG_DIR = DATA_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df, policy = load_clean(DATA_ROOT)
if (DATA_ROOT / "processed" / "SYNTHETIC_DATA.txt").exists():
    print("WARNING: this is FAKE dry-run data. The charts only test the code.\n")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Small": "#9aa5b1", "Medium": "#e0a33a", "Large": "#c4402f"}
df["size_label"] = pd.Categorical(df["size_class"].map(dict(enumerate(CLASS_NAMES))), CLASS_NAMES)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

print(f"{len(df):,} fires, {df.shape[1]} columns, years {df.FIRE_YEAR.min()}–{df.FIRE_YEAR.max()}")

## 1 · How rare are large fires?

In [ ]:
share = df["size_label"].value_counts(normalize=True).reindex(CLASS_NAMES) * 100
by_year = pd.crosstab(df["FIRE_YEAR"], df["size_label"], normalize="index") * 100

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))
a.bar(CLASS_NAMES, share, color=[COLORS[c] for c in CLASS_NAMES])
for i, v in enumerate(share):
    a.text(i, v + 1, f"{v:.1f}%", ha="center")
a.set(title="Share of fires by final size class", ylabel="% of fires")
b.plot(by_year.index, by_year["Large"], marker="o", color=COLORS["Large"])
b.set(title="Large fires (≥300 acres) by year", ylabel="% of fires", xlabel="Fire year")
b.set_ylim(bottom=0)
save(fig, "class_balance")
pd.crosstab(df["FIRE_YEAR"], df["size_label"], margins=True)

## 2 · Fire size is extremely skewed

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(np.log10(df["FIRE_SIZE"] + 0.01), bins=80, color="#5b7083")
for x, label in [(np.log10(10), "10 ac"), (np.log10(300), "300 ac")]:
    ax.axvline(x, color="black", linestyle="--", linewidth=1)
    ax.text(x, ax.get_ylim()[1] * 0.92, f" {label}", fontsize=9)
ax.set(title="Final fire size (log scale), with class boundaries", xlabel="log10(acres)", ylabel="Number of fires")
save(fig, "fire_size_hist")
df["FIRE_SIZE"].describe(percentiles=[.5, .9, .99, .999]).round(2)

## 3 · Where and when

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for label in CLASS_NAMES:  # Large drawn last so it sits on top
    d = df[df["size_label"] == label]
    ax.scatter(d["LONGITUDE"], d["LATITUDE"], s=1 if label != "Large" else 4,
               alpha=0.3 if label != "Large" else 0.8, color=COLORS[label], label=label)
ax.set(title="Fire locations by final size class", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect(1.2)
ax.legend(markerscale=6)
save(fig, "map_by_class")

In [ ]:
monthly = df.groupby("month").agg(fires=("FOD_ID", "size"), large_pct=("size_class", lambda s: 100 * (s == 2).mean()))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(monthly.index, monthly["fires"], color="#c9d1d9", label="All fires")
ax2 = ax.twinx()
ax2.plot(monthly.index, monthly["large_pct"], marker="o", color=COLORS["Large"], label="% that become large")
ax.set(title="Fires per month and share that become large", xlabel="Month", ylabel="Number of fires")
ax2.set_ylabel("% large")
ax.set_xticks(range(1, 13))
save(fig, "seasonality")

In [ ]:
by_state = (df.groupby("STATE", observed=True)
              .agg(fires=("FOD_ID", "size"), large=("size_class", lambda s: int((s == 2).sum())))
              .assign(large_pct=lambda t: (100 * t["large"] / t["fires"]).round(2))
              .sort_values("fires", ascending=False))
by_state

## 4 · Which features separate large fires?
For each numeric feature allowed at discovery, the **AUC** below measures how well that single feature ranks large fires above the rest.
0.5 is useless, 1.0 is perfect, and values below 0.5 mean "lower value = more likely large".
This is a quick screen for the report and for Role C, not a model.

In [ ]:
from sklearn.metrics import roc_auc_score

feats = [c for c in feature_columns(policy, "at_discovery") if pd.api.types.is_numeric_dtype(df[c])]
is_large = (df["size_class"] == 2).astype(int)
rows = []
for c in feats:
    x = df[c]
    ok = x.notna()
    if ok.sum() < 100 or is_large[ok].nunique() < 2:
        continue
    auc = roc_auc_score(is_large[ok], x[ok])
    group = policy.set_index("column").loc[c, "group"]
    rows.append({"feature": c, "group": group, "AUC": round(auc, 3), "strength": round(abs(auc - 0.5), 3),
                 "missing_%": round(100 * (1 - ok.mean()), 1)})
signal = pd.DataFrame(rows).sort_values("strength", ascending=False).reset_index(drop=True)

top = signal.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"], top["strength"], color=[COLORS["Large"] if g == "weather" else "#5b7083" for g in top["group"]])
ax.set(title="Single-feature signal for large fires (|AUC − 0.5|)\nred = weather on discovery day, grey = static",
       xlabel="|AUC − 0.5|")
save(fig, "feature_signal")
signal.head(25)

In [ ]:
# Distribution of the strongest features by class
top_feats = signal["feature"].head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.flat, top_feats):
    data = [df.loc[df["size_label"] == k, c].dropna() for k in CLASS_NAMES]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks([1, 2, 3], CLASS_NAMES)
    ax.set_title(c)
fig.suptitle("Strongest features by final size class (outliers hidden)")
save(fig, "top_features_by_class")

## 5 · Missing data

In [ ]:
miss = (policy[policy["column"].isin(feature_columns(policy, "at_discovery"))]
        .sort_values("missing_pct", ascending=False)[["column", "group", "kind", "missing_pct"]])
print(f"{(miss.missing_pct > 0).sum()} of {len(miss)} at-discovery features have missing values; "
      f"{(miss.missing_pct > 30).sum()} are more than 30% missing.")
miss.head(20)

In [ ]:
# How correlated are the strongest features with each other? (high = redundant)
corr = df[signal["feature"].head(12)].corr()
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation between the 12 strongest features")
save(fig, "feature_correlation")

## 6 · Notes for the report (fill in after running on the real data)
- **Data size:** _N fires, years, states_
- **Class balance:** _x% small, y% medium, z% large; how large-fire share varies by year_
- **Strongest single features:** _top 3 from section 4, and whether they're weather or static_
- **Data quality:** _rows removed per step (notebook 01), features with heavy missingness_
- **Implication for modelling:** _e.g. use class weights; report macro-F1 and large-fire recall, not accuracy_